In [1]:
import sys
import os

# Add project root directory
sys.path.append(os.path.abspath('..'))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

In [3]:
from src.data_loader import load_data

df = load_data('../data/MachineLearningRating_v3.txt')

/Users/new/Desktop/insurance-risk-analytics/src/data_loader.py:10: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=separator)


Data loaded successfully.
Shape: (1000098, 52)


In [4]:
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [5]:
# Claim Frequency
df['HasClaim'] = np.where(df['TotalClaims'] > 0, 1, 0)

# Margin
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

In [6]:
group_a = df[df['Province'] == 'Gauteng']
group_b = df[df['Province'] == 'Western Cape']

In [7]:
t_stat, p_value = ttest_ind(
    group_a['TotalClaims'],
    group_b['TotalClaims'],
    nan_policy='omit'
)

print("P-value:", p_value)

P-value: 0.05632044649871883


In [8]:
contingency_table = pd.crosstab(
    df['Province'],
    df['HasClaim']
)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("P-value:", p_value)

P-value: 5.925510718204677e-19


In [10]:
contingency_table = pd.crosstab(
    df['Province'],
    df['HasClaim']
)

chi2, p1, dof, expected = chi2_contingency(contingency_table)

print("Province p-value:", p1)

Province p-value: 5.925510718204677e-19


In [11]:
zip_a = df[df['PostalCode'] == 2000]
zip_b = df[df['PostalCode'] == 8000]

In [12]:
t_stat, p2 = ttest_ind(
    zip_a['TotalClaims'],
    zip_b['TotalClaims'],
    nan_policy='omit'
)

print("Zip code risk p-value:", p2)

Zip code risk p-value: 0.0006252010270582993


In [13]:
t_stat, p3 = ttest_ind(
    zip_a['Margin'],
    zip_b['Margin'],
    nan_policy='omit'
)

print("Zip code margin p-value:", p3)

Zip code margin p-value: 0.7123247716247569


In [14]:
gender_table = pd.crosstab(
    df['Gender'],
    df['HasClaim']
)

chi2, p4, dof, expected = chi2_contingency(gender_table)

print("Gender risk p-value:", p4)

Gender risk p-value: 0.026570248768437145


In [15]:
results = pd.DataFrame({
    'Hypothesis': [
        'Province Risk',
        'Zip Code Risk',
        'Zip Code Margin',
        'Gender Risk'
    ],
    
    'Test Used': [
        'Chi-Squared',
        'T-Test',
        'T-Test',
        'Chi-Squared'
    ],
    
    'P-Value': [
        p1,
        p2,
        p3,
        p4
    ]
})

results['Decision'] = np.where(
    results['P-Value'] < 0.05,
    'Reject H0',
    'Fail to Reject H0'
)

results

,Hypothesis,Test Used,P-Value,Decision
0,Province Risk,Chi-Squared,5.925511e-19,Reject H0
1,Zip Code Risk,T-Test,6.252010e-04,Reject H0
2,Zip Code Margin,T-Test,7.123248e-01,Fail to Reject H0
3,Gender Risk,Chi-Squared,2.657025e-02,Reject H0
